In [ ]:
# ============================================================
# Cell 1: Setup and Mount Drive
# ============================================================
"""
VGGSound Dataset Download and Extraction
This notebook downloads a subset of VGGSound from HuggingFace,
extracts audio and frames, and saves to Google Drive.
"""

from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Create base directory
BASE_DIR = "/content/drive/MyDrive/spectralbridge_data"
os.makedirs(BASE_DIR, exist_ok=True)
os.makedirs(f"{BASE_DIR}/raw/vggsound", exist_ok=True)
os.makedirs(f"{BASE_DIR}/processed/audio", exist_ok=True)
os.makedirs(f"{BASE_DIR}/processed/images", exist_ok=True)

print(f"✓ Base directory created: {BASE_DIR}")
print(f"✓ Free space: {os.statvfs(BASE_DIR).f_bavail * os.statvfs(BASE_DIR).f_frsize / (1024**3):.1f} GB")



In [ ]:
# ============================================================
# Cell 2: Install Dependencies
# ============================================================
!pip install -q datasets
!apt-get install -y ffmpeg > /dev/null 2>&1

import ffmpeg
from datasets import load_dataset
import tarfile
from pathlib import Path
from tqdm.notebook import tqdm
import shutil

print("✓ Dependencies installed")

In [ ]:
# ============================================================
# Cell 3: Define Selected Categories
# ============================================================
"""
Selected categories (15-20 total):
- Mix of textually-easy and perceptually-rich
"""

SELECTED_CATEGORIES = [
    # Textually easy
    "playing_piano",
    "playing_guitar", 
    "dog_barking",
    "cat_meowing",
    "helicopter",
    "typing_on_keyboard",
    "telephone_ringing",
    
    # Perceptually rich
    "fire_crackling",
    "ocean_waves",
    "thunder",
    "rain_on_surface",
    "wind_blowing",
    "waterfall",
    "hammering",
    "sizzling_food",
    "woodworking",
    "stream_burbling",
]

print(f"Selected {len(SELECTED_CATEGORIES)} categories:")
for cat in SELECTED_CATEGORIES:
    print(f"  - {cat}")


In [ ]:
# ============================================================
# Cell 4: Download Function
# ============================================================
def download_vggsound_subset(num_shards=3):
    """
    Download first N tar files from VGGSound dataset.
    
    Args:
        num_shards: Number of tar files to download (out of 20 total)
    """
    print(f"\n{'='*60}")
    print(f"Downloading {num_shards} VGGSound tar files...")
    print(f"{'='*60}\n")
    
    # Load dataset in streaming mode to get file list
    dataset = load_dataset(
        "Loie/VGGSound", 
        split="train",
        streaming=True
    )
    
    # The dataset is in webdataset format (tar files)
    # We'll download to a temporary location first
    download_dir = "/content/vggsound_download"
    os.makedirs(download_dir, exist_ok=True)
    
    print("Loading dataset iterator...")
    
    # Download by iterating through samples
    # Each tar file contains multiple MP4s
    downloaded_count = 0
    category_counts = {cat: 0 for cat in SELECTED_CATEGORIES}
    
    for idx, sample in enumerate(tqdm(dataset, desc="Processing samples")):
        # Stop after reasonable number of samples
        # Each tar has ~1000-2000 videos
        if downloaded_count >= num_shards * 1500:
            break
        
        try:
            # Get video data
            if 'mp4' not in sample or sample['mp4'] is None:
                continue
            
            # Get metadata
            key = sample.get('__key__', f'video_{idx}')
            
            # Try to infer category from key
            # VGGSound keys usually contain category info
            category = None
            for cat in SELECTED_CATEGORIES:
                if cat.replace('_', ' ') in key.lower() or cat in key.lower():
                    category = cat
                    break
            
            # If we can't determine category, skip for now
            # (we'll handle this better in next iteration)
            if category is None:
                continue
            
            # Save MP4
            category_dir = Path(BASE_DIR) / "raw" / "vggsound" / category
            category_dir.mkdir(parents=True, exist_ok=True)
            
            mp4_path = category_dir / f"{key}.mp4"
            
            # Write MP4 bytes
            with open(mp4_path, 'wb') as f:
                f.write(sample['mp4'])
            
            category_counts[category] += 1
            downloaded_count += 1
            
            # Stop if we have enough samples
            if all(count >= 50 for count in category_counts.values()):
                print("\n✓ Reached target: 50 samples per category")
                break
                
        except Exception as e:
            continue
    
    print(f"\n{'='*60}")
    print("Download Summary:")
    print(f"{'='*60}")
    for cat, count in category_counts.items():
        print(f"{cat:30s}: {count:3d} videos")
    print(f"{'='*60}")
    print(f"Total downloaded: {downloaded_count}")
    
    return category_counts

In [ ]:
# ============================================================
# Cell 5: Extract Audio and Frame Function
# ============================================================
def extract_audio_and_frame(mp4_path, output_audio_path, output_image_path):
    """
    Extract audio as WAV and middle frame as JPG from MP4.
    
    Args:
        mp4_path: Path to input MP4
        output_audio_path: Path for output WAV file
        output_image_path: Path for output JPG file
    """
    try:
        # Get video duration to find middle frame
        probe = ffmpeg.probe(str(mp4_path))
        duration = float(probe['streams'][0]['duration'])
        middle_time = duration / 2
        
        # Extract audio (16kHz mono WAV)
        (
            ffmpeg
            .input(str(mp4_path))
            .output(str(output_audio_path), 
                   acodec='pcm_s16le',
                   ac=1,
                   ar='16000')
            .overwrite_output()
            .run(quiet=True)
        )
        
        # Extract middle frame (518x518 for DINOv2)
        (
            ffmpeg
            .input(str(mp4_path), ss=middle_time)
            .filter('scale', 518, 518)
            .output(str(output_image_path), vframes=1)
            .overwrite_output()
            .run(quiet=True)
        )
        
        return True
    except Exception as e:
        print(f"Error processing {mp4_path}: {e}")
        return False

In [ ]:
# ============================================================
# Cell 6: Process All Videos
# ============================================================
def process_all_videos():
    """
    Process all downloaded MP4s: extract audio and frames.
    """
    print(f"\n{'='*60}")
    print("Processing all videos...")
    print(f"{'='*60}\n")
    
    raw_dir = Path(BASE_DIR) / "raw" / "vggsound"
    audio_dir = Path(BASE_DIR) / "processed" / "audio"
    image_dir = Path(BASE_DIR) / "processed" / "images"
    
    stats = {"success": 0, "failed": 0}
    
    # Process each category
    for category in SELECTED_CATEGORIES:
        category_raw = raw_dir / category
        
        if not category_raw.exists():
            continue
        
        # Create output directories
        (audio_dir / category).mkdir(parents=True, exist_ok=True)
        (image_dir / category).mkdir(parents=True, exist_ok=True)
        
        # Process each MP4
        mp4_files = list(category_raw.glob("*.mp4"))
        
        print(f"\nProcessing {category}: {len(mp4_files)} videos")
        
        for mp4_path in tqdm(mp4_files, desc=category):
            video_id = mp4_path.stem
            
            audio_path = audio_dir / category / f"{video_id}.wav"
            image_path = image_dir / category / f"{video_id}.jpg"
            
            # Skip if already processed
            if audio_path.exists() and image_path.exists():
                stats["success"] += 1
                continue
            
            # Extract
            success = extract_audio_and_frame(mp4_path, audio_path, image_path)
            
            if success:
                stats["success"] += 1
            else:
                stats["failed"] += 1
    
    print(f"\n{'='*60}")
    print("Processing Summary:")
    print(f"{'='*60}")
    print(f"Successful: {stats['success']}")
    print(f"Failed: {stats['failed']}")
    print(f"Success rate: {stats['success']/(stats['success']+stats['failed'])*100:.1f}%")
    
    return stats

In [ ]:
# ============================================================
# Cell 7: RUN DOWNLOAD
# ============================================================
print("Starting VGGSound download...")
print("This will take ~1-2 hours. You can close this tab and come back.")
print("Progress is saved to Google Drive automatically.\n")

category_counts = download_vggsound_subset(num_shards=3)



In [ ]:
# ============================================================
# Cell 8: RUN EXTRACTION
# ============================================================
print("\nStarting audio and frame extraction...")
print("This will take ~30-60 minutes.\n")

processing_stats = process_all_videos()

print("\n✓ VGGSound data ready!")
print(f"✓ Location: {BASE_DIR}")

In [ ]:
# ============================================================
# Cell 9: Verify Data
# ============================================================
def verify_data():
    """Check what we have."""
    audio_dir = Path(BASE_DIR) / "processed" / "audio"
    image_dir = Path(BASE_DIR) / "processed" / "images"
    
    print(f"\n{'='*60}")
    print("DATA VERIFICATION")
    print(f"{'='*60}\n")
    
    for category in SELECTED_CATEGORIES:
        audio_count = len(list((audio_dir / category).glob("*.wav"))) if (audio_dir / category).exists() else 0
        image_count = len(list((image_dir / category).glob("*.jpg"))) if (image_dir / category).exists() else 0
        
        print(f"{category:30s}: {audio_count:3d} audio | {image_count:3d} images")
    
    total_audio = sum(len(list(cat.glob("*.wav"))) for cat in audio_dir.glob("*") if cat.is_dir())
    total_images = sum(len(list(cat.glob("*.jpg"))) for cat in image_dir.glob("*") if cat.is_dir())
    
    print(f"\n{'='*60}")
    print(f"TOTAL: {total_audio} audio files | {total_images} images")
    print(f"{'='*60}")

verify_data()